In [ ]:
# Locate the project root (where config.yaml lives) and set as cwd
import os
from pathlib import Path
for _parent in [Path.cwd(), Path.cwd().parent]:
    if (_parent / 'config.yaml').exists():
        os.chdir(_parent)
        break
print('Working directory:', os.getcwd())


# Step 08: TCGA survival validation


Converts TCGA CRC (COAD + READ) counts to log2(TPM+1) using GENCODE gene lengths, then performs Kaplan-Meier, Cox regression, and prognostic screening.


Download TCGA data first: `bash scripts/download/download_tcga.sh`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import yaml
from scripts.tcga import counts_to_tpm, cox_regression, km_plot, screen_degs, plot_cox_volcano
from utils.helpers import ensure_dir
import gzip, pandas as pd

cfg = yaml.safe_load(open('config.yaml'))
ensure_dir('Figures/08_tcga')
tpm = counts_to_tpm('Data/TCGA/CRC_raw_counts.tsv.gz', cfg['gencode_gtf'],
                    'Data/TCGA/CRC_log2tpm.tsv.gz')
with gzip.open('Data/TCGA/CRC_survival.tsv.gz', 'rt') as f:
    surv = pd.read_csv(f, sep='\t', index_col=0)
surv.index = [s[:12] for s in surv.index]

cox_rows = []
for g in ['PCOLCE2', 'GDF15', 'HMMR']:
    r = cox_regression(g, tpm, surv)
    cox_rows.append(r)
    print(g, 'HR_mv=%.3f p_mv=%.4f' % (r['HR_mv'], r['p_mv']))
    km_plot(g, tpm, surv, f'Figures/08_tcga/KM_{g}.png')
pd.DataFrame(cox_rows).to_csv('Data/tables/cox_results.tsv', sep='\t', index=False)
screening_df = screen_degs('Data/tables/deseq2_significant_siKRAS_vs_siScr.tsv', tpm, surv,
            'Data/tables/cox_screening.tsv')
plot_cox_volcano(screening_df, 'Figures/08_tcga/cox_volcano.png')
